In [8]:
import pandas as pd
import numpy as np
import joblib
import json

DIRS = {
    'models': '../models',
    'ltr_data': '../data/processed/ltr_datasets'
}

# Load TUNED XGBoost model (best performer from notebook 07)
# Falls back to baseline if tuned model not found
import os
tuned_path = f"{DIRS['models']}/xgboost_tuned.pkl"
baseline_path = f"{DIRS['models']}/xgboost_classifier.pkl"

if os.path.exists(tuned_path):
    xgb_model = joblib.load(tuned_path)
    print(f"Loaded TUNED model: {tuned_path}")
    model_version = "tuned"
else:
    xgb_model = joblib.load(baseline_path)
    print(f"Tuned model not found, loaded BASELINE: {baseline_path}")
    model_version = "baseline"

# Get feature names from model
model_feature_columns = xgb_model.get_booster().feature_names
if not model_feature_columns:
    with open(f"{DIRS['models']}/feature_columns.json", 'r') as f:
        saved_meta = json.load(f)
    model_feature_columns = saved_meta['features'] if isinstance(saved_meta, dict) else saved_meta

if not model_feature_columns:
    raise ValueError("Cannot determine feature columns.")

print(f"Model expects {len(model_feature_columns)} features")

candidate_pool = pd.read_csv(f"{DIRS['ltr_data']}/employee_candidate_pool.csv")
if "Employee_Name" not in candidate_pool.columns:
    candidate_pool["Employee_Name"] = candidate_pool["Employee_ID"].astype(str)

ltr_train = pd.read_csv(f"{DIRS['ltr_data']}/ltr_train_dataset.csv")
print(f"Loaded {len(ltr_train)} training rows and {len(candidate_pool)} candidates")
print(f"Active model: XGBoost Classifier ({model_version.upper()})")


Loaded TUNED model: ../models/xgboost_tuned.pkl
Model expects 56 features
Loaded 36605 training rows and 49 candidates
Active model: XGBoost Classifier (TUNED)


In [9]:
# Build employee feature lookup from training data
# Uses defensive .agg() to handle missing columns gracefully

relevant = ltr_train[ltr_train['relevance'] == 1].copy()
print(f"Found {len(relevant)} relevant training rows")
print(f"Relevant columns: {list(relevant.columns)[:20]}...")

# Build aggregation dict dynamically based on what columns exist
agg_dict = {'Task_ID': 'count'}  # Always available

if 'Estimated_Planned_Hours' in relevant.columns:
    agg_dict['Estimated_Planned_Hours'] = 'mean'

if 'Task_Skill_Count' in relevant.columns:
    agg_dict['Task_Skill_Count'] = 'mean'

if 'Task_Text_Length' in relevant.columns:
    agg_dict['Task_Text_Length'] = 'mean'

if 'Days_To_Deadline' in relevant.columns:
    agg_dict['Days_To_Deadline'] = 'mean'

print(f"Aggregating on: {list(agg_dict.keys())}")

emp_features = relevant.groupby('Employee_ID').agg(agg_dict).reset_index()

# Rename Task_ID count to employee_historical_task_count
emp_features = emp_features.rename(columns={'Task_ID': 'employee_historical_task_count'})

# Rename other columns to expected names
rename_map = {}
if 'Estimated_Planned_Hours' in emp_features.columns:
    rename_map['Estimated_Planned_Hours'] = 'employee_historical_avg_planned_hours'
if 'Task_Skill_Count' in emp_features.columns:
    rename_map['Task_Skill_Count'] = 'employee_historical_avg_skill_count'
if 'Task_Text_Length' in emp_features.columns:
    rename_map['Task_Text_Length'] = 'employee_historical_avg_task_text_length'
if 'Days_To_Deadline' in emp_features.columns:
    rename_map['Days_To_Deadline'] = 'employee_historical_avg_deadline_days'

emp_features = emp_features.rename(columns=rename_map)

# Unique projects
if 'Project_Name' in relevant.columns:
    proj_unique = relevant.groupby('Employee_ID')['Project_Name'].nunique().reset_index()
    proj_unique.columns = ['Employee_ID', 'employee_historical_unique_projects']
    emp_features = emp_features.merge(proj_unique, on='Employee_ID', how='left')
else:
    emp_features['employee_historical_unique_projects'] = 1

emp_features['employee_historical_project_count'] = emp_features['employee_historical_unique_projects']

# Log task count
emp_features['employee_log_task_count'] = np.log1p(emp_features['employee_historical_task_count'])

# Employee skill profiles
skill_cols_in_data = [c for c in relevant.columns if c.startswith('Skill_') and not c.startswith('Employee_Profile_')]
if skill_cols_in_data:
    emp_skill_profile = relevant.groupby('Employee_ID')[skill_cols_in_data].mean().reset_index()
    rename_skill = {c: c.replace('Skill_', 'Employee_Profile_Skill_') for c in skill_cols_in_data}
    emp_skill_profile = emp_skill_profile.rename(columns=rename_skill)
    emp_features = emp_features.merge(emp_skill_profile, on='Employee_ID', how='left')

print(f"Built employee features for {len(emp_features)} employees")
print(f"Final columns ({len(emp_features.columns)}): {list(emp_features.columns)}")

# CRITICAL: Verify required columns exist
required_for_inference = ['employee_historical_task_count', 'employee_historical_unique_projects']
missing = [c for c in required_for_inference if c not in emp_features.columns]
if missing:
    raise ValueError(f"Missing required columns in emp_features: {missing}")
print("Required columns verified.")



Found 2425 relevant training rows
Relevant columns: ['Task_ID', 'Employee_ID', 'relevance', 'Hours_Spent', 'FLAG_LEAKAGE_Timesheet_Work_Logs', 'Task_Priority', 'Estimated_Planned_Hours', 'FLAG_LEAKAGE_Actual_Hours_Spent', 'FLAG_LEAKAGE_Timesheet_Logs_Count', 'FLAG_LEAKAGE_Task_Stage', 'FLAG_LEAKAGE_All_Collaborating_Employees', 'Task_Text_Length', 'Task_Word_Count', 'Task_Description_Length', 'Task_Name_Length', 'Has_Task_Description', 'Created_Year', 'Created_Month', 'Created_DayOfWeek', 'Created_Quarter']...
Aggregating on: ['Task_ID', 'Estimated_Planned_Hours', 'Task_Skill_Count', 'Task_Text_Length', 'Days_To_Deadline']
Built employee features for 49 employees
Final columns (18): ['Employee_ID', 'employee_historical_task_count', 'employee_historical_avg_planned_hours', 'employee_historical_avg_skill_count', 'employee_historical_avg_task_text_length', 'employee_historical_avg_deadline_days', 'employee_historical_unique_projects', 'employee_historical_project_count', 'employee_log_tas

In [10]:
SKILL_COLS = [c for c in model_feature_columns if c.startswith('Skill_') and not c.startswith('Employee_Profile_')]
SKILL_KEYWORDS = {
    'Skill_Odoo_ERP_Development': ['odoo', 'erp', 'addon'],
    'Skill_Database_Management': ['database', 'sql'],
    'Skill_Server_Administration': ['server', 'vps', 'ssl', 'deploy'],
    'Skill_Project_Management': ['meeting', 'plan', 'scrum'],
    'Skill_Software_Testing': ['test', 'uat', 'qa', 'bug'],
    'Skill_Web_Development': ['web', 'frontend', 'backend'],
    'Skill_Client_and_Functional_Support': ['client', 'support', 'functional'],
    'Skill_Documentation': ['document', 'report', 'blueprint'],
    'Skill_Training_and_Mentorship': ['train', 'mentor', 'teach']
}

def detect_task_skills(task_text):
    desc_lower = str(task_text).lower()
    skills = {}
    for skill_col, keywords in SKILL_KEYWORDS.items():
        skills[skill_col] = int(any(kw in desc_lower for kw in keywords))
    return skills


def extract_task_features_real(task_df, candidates_df, emp_features_df):
    out = candidates_df[['Employee_ID', 'Employee_Name']].copy()
    n = len(out)
    
    task_desc = str(task_df['Task_Description'].iloc[0]) if 'Task_Description' in task_df.columns else ''
    task_title = str(task_df['Task_Title'].iloc[0]) if 'Task_Title' in task_df.columns else ''
    
    out['Task_Text_Length'] = len(task_desc)
    out['Task_Word_Count'] = len(task_desc.split())
    out['Task_Description_Length'] = len(task_desc)
    out['Task_Name_Length'] = len(task_title)
    out['Has_Task_Description'] = 1 if task_desc else 0
    
    if 'Estimated_Planned_Hours' in task_df.columns:
        hours = float(task_df['Estimated_Planned_Hours'].iloc[0])
    else:
        hours = 0.0
    out['Estimated_Planned_Hours'] = hours
    out['Planned_Hours_Log'] = np.log1p(hours)
    out['Planned_Task_Size_Small'] = 1 if hours <= 8 else 0
    out['Planned_Task_Size_Medium'] = 1 if 8 < hours <= 40 else 0
    out['Planned_Task_Size_Large'] = 1 if hours > 40 else 0
    
    if 'Days_To_Deadline' in task_df.columns:
        out['Days_To_Deadline'] = float(task_df['Days_To_Deadline'].iloc[0])
        out['Has_Deadline'] = 1
    else:
        out['Days_To_Deadline'] = np.nan
        out['Has_Deadline'] = 0
    
    task_skills = detect_task_skills(task_desc + ' ' + task_title)
    for skill_col, val in task_skills.items():
        out[skill_col] = val
    out['Task_Skill_Count'] = sum(task_skills.values())
    
    import datetime
    today = datetime.datetime.now()
    out['Created_Year'] = today.year
    out['Created_Month'] = today.month
    out['Created_DayOfWeek'] = today.weekday()
    out['Created_Quarter'] = (today.month - 1) // 3 + 1
    
    priority = task_df['Task_Priority'].iloc[0] if 'Task_Priority' in task_df.columns else 'Low'
    out['Task_Priority_Low'] = 1 if priority == 'Low' else 0
    out['Task_Priority_Normal'] = 1 if priority != 'Low' else 0
    
    # Map employee features with safe defaults
    emp_lookup = emp_features_df.set_index('Employee_ID')
    
    # CRITICAL: Ensure required columns exist with safe defaults
    if 'employee_historical_task_count' not in emp_features_df.columns:
        print("WARNING: employee_historical_task_count missing, using median fallback")
        emp_features_df['employee_historical_task_count'] = 10
    
    if 'employee_historical_unique_projects' not in emp_features_df.columns:
        print("WARNING: employee_historical_unique_projects missing, using 1")
        emp_features_df['employee_historical_unique_projects'] = 1
    
    # ALWAYS map these two because they are used in calculations directly below
    out['employee_historical_task_count'] = out['Employee_ID'].map(emp_lookup['employee_historical_task_count']).fillna(10)
    out['employee_historical_unique_projects'] = out['Employee_ID'].map(emp_lookup['employee_historical_unique_projects']).fillna(1)

    for col in emp_features_df.columns:
        if col == 'Employee_ID' or col in ['employee_historical_task_count', 'employee_historical_unique_projects']:
            continue
        if col in model_feature_columns:
            default_val = emp_features_df[col].median() if pd.api.types.is_numeric_dtype(emp_features_df[col]) else 0
            out[col] = out['Employee_ID'].map(emp_lookup[col]).fillna(default_val)
    
    # Project experience calculation - now safe
    out['employee_project_task_count'] = (
        out['employee_historical_task_count'] / out['employee_historical_unique_projects'].clip(lower=1)
    )
    out['employee_has_project_experience'] = (out['employee_project_task_count'] > 0).astype(int)
    
    # Skill match features
    if SKILL_COLS:
        emp_skill_match = pd.Series(0.0, index=out.index)
        for skill_col in SKILL_COLS:
            emp_profile_col = skill_col.replace('Skill_', 'Employee_Profile_Skill_')
            if emp_profile_col in out.columns and skill_col in out.columns:
                emp_skill_match = emp_skill_match + (out[skill_col].astype(float) * out[emp_profile_col].astype(float))
        out['employee_task_skill_match_count'] = emp_skill_match.astype(int)
        out['employee_task_skill_match_ratio'] = emp_skill_match / max(out['Task_Skill_Count'].max(), 1)
        out['employee_has_matching_skill'] = (out['employee_task_skill_match_count'] > 0).astype(int)
    else:
        out['employee_task_skill_match_count'] = 0
        out['employee_task_skill_match_ratio'] = 0.0
        out['employee_has_matching_skill'] = 0
    
    for col in model_feature_columns:
        if col not in out.columns:
            out[col] = np.nan
    
    return out



In [11]:
def recommend_employees_for_task_real(raw_task_dict, candidate_df, model, 
                                        emp_features_df, expected_features, top_k=5):
    print(f"Analyzing Task: {raw_task_dict.get('Task_Title', 'Unknown Task')}...\n")
    
    task_df = pd.DataFrame([raw_task_dict])
    inference_df = extract_task_features_real(task_df, candidate_df, emp_features_df)
    
    X_inference = inference_df[expected_features].copy()
    
    inference_df['prediction_score'] = model.predict_proba(X_inference)[:, 1]
    
    top_candidates = inference_df.sort_values(by='prediction_score', ascending=False).head(top_k)
    
    results = top_candidates[['Employee_ID', 'Employee_Name', 'prediction_score']].copy()
    results['Rank'] = range(1, top_k + 1)
    results['Match_Confidence'] = (results['prediction_score'] * 100).round(1).astype(str) + '%'
    
    return results[['Rank', 'Employee_ID', 'Employee_Name', 'Match_Confidence']]



In [16]:
new_task = {
"Task_ID": "TSK-6099",
    "Project_ID": "PRJ-550",
    "Task_Title": "Configure CI/CD Pipeline for Microservices",
    "Task_Description": "Set up automated testing and deployment pipelines using GitHub Actions for the inventory and order management microservices.",
    "Required_Skills": "Docker, CI/CD, GitHub Actions, AWS",
    "Estimated_Planned_Hours": 32.0,
    "Days_To_Deadline": 14
}

recommendations = recommend_employees_for_task_real(
    raw_task_dict=new_task,
    candidate_df=candidate_pool,
    model=xgb_model,
    emp_features_df=emp_features,
    expected_features=model_feature_columns,
    top_k=5
)

print("TOP EMPLOYEE RECOMMENDATIONS (XGBOOST - Real Features)")
print("=" * 60)
print(recommendations.to_string(index=False))
print("=" * 60)



Analyzing Task: Configure CI/CD Pipeline for Microservices...

TOP EMPLOYEE RECOMMENDATIONS (XGBOOST - Real Features)
 Rank Employee_ID Employee_Name Match_Confidence
    1       EMP-9         EMP-9            43.1%
    2      EMP-12        EMP-12            18.9%
    3      EMP-41        EMP-41            15.0%
    4       EMP-5         EMP-5            15.0%
    5      EMP-42        EMP-42            14.7%


In [13]:
# Display model metadata for reproducibility
import os
metadata_path = f"{DIRS['models']}/optuna_tuning_metadata.json"
if os.path.exists(metadata_path):
    with open(metadata_path, 'r') as f:
        tuning_meta = json.load(f)
    print("=" * 60)
    print("MODEL METADATA (from Optuna tuning)")
    print("=" * 60)
    print(f"Tuning timestamp: {tuning_meta.get('tuning_timestamp', 'N/A')}")
    print(f"Framework: {tuning_meta.get('framework', 'N/A')}")
    print(f"Objective metric: {tuning_meta.get('objective_metric', 'N/A')}")
    print(f"Test set status: {tuning_meta.get('test_set_status', 'N/A')}")
    print(f"\nXGBoost best NDCG@5 (test): {tuning_meta.get('xgb_test_ndcg5', 'N/A'):.4f}")
    print(f"XGBoost improvement over baseline: {tuning_meta.get('xgb_test_improvement_over_baseline', 0)*100:+.2f}%")
    print(f"\nBest hyperparameters:")
    for k, v in tuning_meta.get('xgb_best_params', {}).items():
        print(f"  {k}: {v}")
    print("=" * 60)
else:
    print(f"No tuning metadata found at {metadata_path}")


MODEL METADATA (from Optuna tuning)
Tuning timestamp: 2026-09-20 12:25:04
Framework: Optuna
Objective metric: NDCG@5 on validation set
Test set status: UNTOUCHED during optimization

XGBoost best NDCG@5 (test): 0.8443
XGBoost improvement over baseline: +2.76%

Best hyperparameters:
  n_estimators: 500
  learning_rate: 0.16355493474708033
  max_depth: 7
  min_child_weight: 5
  subsample: 0.691361532073299
  colsample_bytree: 0.6075350817982986
  gamma: 0.16796205152370952
  reg_alpha: 0.7032782852617623
  reg_lambda: 1.8620154008254997
